In [ ]:
import numpy as np


def generate_and_fit_polynomial(
    total_points=1580,
    desired_fit_degree=4,
    periodicity=24,
    noise_std=0.05,
    seed=42,
    num_context=20,
    skip=700,
):
    np.random.seed(seed)

    def ground_truth_polynomial(x_scaled):
        # Higher-degree nonlinear structure
        p1 = 0.3 * x_scaled**4
        p2 = -0.2 * x_scaled**5
        p3 = 0.1 * x_scaled**6
        periodic = 0.3 * np.sin(2 * np.pi * x_scaled * periodicity)

        # Combine components
        values = p1 + p2 + p3 + periodic
        values -= values.min()
        values /= values.max()
        return values

    # --- FIX 1: Scale x to [-1, 1] ---
    x_raw = np.linspace(500, 2000, total_points)
    x_vals = np.linspace(
        -1, 1, total_points
    )  # fit over [-1, 1] for numerical stability

    # --- FIX 2: Generate target y ---
    y_clean = ground_truth_polynomial(x_vals)
    y_vals = y_clean + np.random.normal(0, noise_std, size=total_points)

    coeffs_and_errors = []
    y_true_vals = []
    # --- Fit and evaluate ---
    for i in range(skip, skip + num_context, 1):
        x_obs = x_vals[:i]
        y_obs = y_vals[:i]

        coeffs = np.polyfit(x_obs, y_obs, desired_fit_degree)
        poly = np.poly1d(coeffs)

        x_next = x_vals[i]
        y_pred = poly(x_next)
        y_true = y_vals[i]

        error = y_pred - y_true
        coeffs_and_errors.append(np.concatenate([coeffs, [error]]))
        y_true_vals.append(y_true)

    return x_raw, y_vals, np.array(coeffs_and_errors), np.array(y_true_vals)


# This forward is very slow, aboyt 1.5 hours for railway dataset
# context_fit = batch['context']['x'].squeeze(0)
# target_fit = batch['target']['x'].squeeze(0)

# context = batch['context']['y'].squeeze(0)
# target = batch['target']['y']
# outputs = []
# for idx in range(target.shape[1]):
#     # Get the right context and target for the current index

#     self.fit(context_fit[idx::24,:,], target_fit[idx::24,:,])
#     # if self.aux_context==None:
#     #     desired_fit_degree = 4
#     #     _, _, context_aux, y_aux = generate_and_fit_polynomial(desired_fit_degree=desired_fit_degree)

#     #     # Generate zeros
#     #     context_aux = np.concatenate([np.zeros((context_aux.shape[0], context_fit.shape[1] - desired_fit_degree - 2)), context_aux], axis=1)
#     #     self.aux_context = torch.Tensor(context_aux).to(batch['context']['x'].device)
#     #     self.y_aux = torch.Tensor(y_aux.reshape(-1,1)).to(batch['context']['x'].device)


#     # context_fit = torch.cat([self.aux_context, context_fit], dim=0)
#     # target_fit = torch.cat([self.y_aux, target_fit], dim=0)


#     output = self.backbone.predict(context[idx].unsqueeze(0).cpu())
#     outputs.append(output.reshape(-1, 1))

# outputs = np.concatenate(outputs, axis=0)#     outputs.append(output.reshape(-1, 1))

# outputs = np.concatenate(outputs, axis=0)

In [11]:
# Loader pytorch lightning chckpoint 
import torch
checkpoint = '/home/levtel/projects/TabPHM/checkpoints/autoencoders/battery/epoch004-val_loss0.00.ckpt'


# Use the context manager to temporarily allow this class

checkpoint = torch.load(checkpoint, weights_only=True)

print("Checkpoint loaded successfully and securely!")



UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL lmetk.loss.default.MSELoss was not an allowed global by default. Please use `torch.serialization.add_safe_globals([lmetk.loss.default.MSELoss])` or the `torch.serialization.safe_globals([lmetk.loss.default.MSELoss])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more abou

In [60]:
from lmetk.loss.default import MSELoss
import torch
checkpoint_path = '/home/levtel/projects/TabPHM/checkpoints/autoencoders/battery/epoch004-val_loss0.00-v1.ckpt'

# Define a list of classes you trust
safe_globals = [MSELoss]

# Use the context manager to temporarily allow this class during loading
with torch.serialization.safe_globals(safe_globals):
    checkpoint = torch.load(checkpoint_path, weights_only=False)

print("Checkpoint loaded successfully and securely!")

Checkpoint loaded successfully and securely!


In [62]:
checkpoint.keys() #['hyper_parameters']

dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'hparams_name', 'hyper_parameters', 'datamodule_hparams_name', 'datamodule_hyper_parameters'])

In [34]:
from typing import List, Optional, Tuple
import awkward as ak
import torch
import numpy as np


class JaggedAutoregressiveSequencer:
    """
    Autoregressive sequencer for jagged (awkward) arrays with padding support.

    Features and time_features are ragged sequences (ak.Array) where each top-level
    element is a time-series of variable length. This sequencer extracts
    sliding windows with autoregressive structure:
    
        - Input window (seq_x): length = seq_len
        - Output window (seq_y): length = label_len + pred_len
        - Overlap between encoder and decoder controlled by label_len
        - Stride determines how windows move along the sequence
    
    If a sequence is too short, we pad it with None values so that every
    sequence produces windows of the same shape.

    Args
    ----
    features : ak.Array
        Ragged time-series data, e.g. shape (n_samples, variable_time, feat_dim?)
    time_features : ak.Array or None
        Ragged time-based features with same outer length as features.
    seq_len : int
        Length of the encoder input sequence.
    label_len : int
        Overlap length between encoder and decoder.
    pred_len : int
        Prediction horizon length.
    stride : int
        Step size between consecutive windows.
    device : torch.device or None
        Device to place torch tensors on.
    dtype : torch.dtype
        Tensor dtype.
    """

    def __init__(
        self,
        features: ak.Array,
        time_features: Optional[ak.Array],
        seq_len: int,
        label_len: int,
        pred_len: int,
        stride: int = 1,
        device: Optional[torch.device] = None,
        dtype: torch.dtype = torch.float32,
    ):
        # --- validation
        assert isinstance(features, ak.Array), "features must be an awkward.Array"
        if time_features is not None:
            assert isinstance(time_features, ak.Array), "time_features must be an awkward.Array"
            assert len(time_features) == len(features), "features and time_features must align"

        self.features = features
        self.time_features = time_features
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.stride = stride
        self.device = device or torch.device("cpu")
        self.dtype = dtype

        # precompute window index list
        self._windows: List[Tuple[int, int]] = []
        self._precompute_windows()

    def _precompute_windows(self):
        """
        Build a list of all valid (sample_idx, start_time) pairs.

        Each entry defines one sliding window inside one top-level sequence.
        If the sequence is too short, we still add one window and will pad it later.
        """
        self._windows = []
        n_samples = len(self.features)
        for i in range(n_samples):
            # sequence length
            L = ak.num(self.features[i], axis=0)

            # at least one window should be created (even if padded)
            if L <= 0:
                continue

            max_t_inclusive = max(0, L - (self.seq_len + self.pred_len))
            t = 0
            while t <= max_t_inclusive:
                self._windows.append((i, t))
                t += self.stride

            # If the sequence was shorter than needed, add one padded window
            if L < (self.seq_len + self.pred_len):
                self._windows.append((i, 0))

    def __len__(self):
        return len(self._windows)

    def _ak_slice_to_tensor(self, ak_subarray, target_len: int) -> torch.Tensor:
        """
        Slice a ragged subarray, pad it to target length, convert to torch.Tensor.

        Args
        ----
        ak_subarray : ak.Array
            The ragged subarray to slice.
        target_len : int
            Desired padded length along time dimension.

        Returns
        -------
        torch.Tensor
            Shape (target_len, feat_dim) with NaN where padded.
        """
        # pad with None to reach target length
        ak_padded = ak.pad_none(ak_subarray, target_len, axis=0, clip=True)

        # convert to list -> numpy
        lst = ak.to_list(ak_padded)
        arr = np.array(lst, dtype=np.float32)

        # replace None with np.nan
        arr = np.where(arr == None, np.nan, arr).astype(np.float32)

        # if it's 1D, make it (T,1)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        return torch.from_numpy(arr).to(dtype=self.dtype, device=self.device)

    def _get_window_tensors_for(self, sample_idx: int, t: int):
        """
        Extract one window (seq_x, seq_y, seq_x_time, seq_y_time) from a sample.
        Pads if the sequence is too short.
        """
        # compute start/end indices
        x_start = t
        x_end = t + self.seq_len
        y_start = t + self.seq_len - self.label_len
        y_end = y_start + (self.label_len + self.pred_len)

        # slice awkward arrays
        feat_sample = self.features[sample_idx]
        seq_x_ak = feat_sample[x_start:x_end]
        seq_y_ak = feat_sample[y_start:y_end]

        seq_x = self._ak_slice_to_tensor(seq_x_ak, self.seq_len)
        seq_y = self._ak_slice_to_tensor(seq_y_ak, self.label_len + self.pred_len)

        # handle time features
        if self.time_features is not None:
            tf_sample = self.time_features[sample_idx]
            seq_x_time = self._ak_slice_to_tensor(tf_sample[x_start:x_end], self.seq_len)
            seq_y_time = self._ak_slice_to_tensor(tf_sample[y_start:y_end], self.label_len + self.pred_len)
        else:
            seq_x_time = None
            seq_y_time = None

        return seq_x, seq_y, seq_x_time, seq_y_time

    def sequences_batch(self, batch_idx: List[int]):
        """
        Collect multiple windows into one batch.

        Args
        ----
        batch_idx : List[int]
            Indices of windows to retrieve.

        Returns
        -------
        (seq_x, seq_y, seq_x_time, seq_y_time) : tuple of torch.Tensor
            Batched tensors with shapes:
                seq_x: (B, seq_len, feat_dim)
                seq_y: (B, label_len+pred_len, feat_dim)
        """
        seq_x_list, seq_y_list, seq_x_time_list, seq_y_time_list = [], [], [], []

        for win_idx in batch_idx:
            sample_idx, t = self._windows[win_idx]
            seq_x, seq_y, seq_x_time, seq_y_time = self._get_window_tensors_for(sample_idx, t)

            seq_x_list.append(seq_x)
            seq_y_list.append(seq_y)
            if self.time_features is not None:
                seq_x_time_list.append(seq_x_time)
                seq_y_time_list.append(seq_y_time)

        seq_x_b = torch.stack(seq_x_list, dim=0)
        seq_y_b = torch.stack(seq_y_list, dim=0)
        seq_x_time_b = torch.stack(seq_x_time_list, dim=0) if self.time_features is not None else None
        seq_y_time_b = torch.stack(seq_y_time_list, dim=0) if self.time_features is not None else None

        return seq_x_b, seq_y_b, seq_x_time_b, seq_y_time_b

    def get_window_info(self, idx: int):
        """Return (sample_idx, t) for a window index."""
        return self._windows[idx]


In [35]:
# test_jagged.py
import awkward as ak
import torch

features = ak.Array([[0, 1, 2], [3, 4], [5], [6, 7, 8, 9]])
# dummy time_features (same shape)
timef = ak.Array([[0,0,0], [0,0], [0], [0,0,0,0]])

dataset = JaggedSlidingWindowDataset(
    data_dict={"features": features, "time_features": timef},
    seq_len=2,
    label_len=1,
    pred_len=1,
    stride=1,
    device=torch.device("cpu"),
)

print("Number of windows:", len(dataset))
# enumerate windows and their tensors
for i in range(len(dataset)):
    seq_x, seq_y, seq_x_time, seq_y_time = dataset[i]
    print("window", i)
    print("seq_x:", seq_x.shape, seq_x)
    print("seq_y:", seq_y.shape, seq_y)


Number of windows: 7
window 0
seq_x: torch.Size([1, 2, 1]) tensor([[[0.],
         [1.]]])
seq_y: torch.Size([1, 2, 1]) tensor([[[1.],
         [2.]]])
window 1
seq_x: torch.Size([1, 2, 1]) tensor([[[3.],
         [4.]]])
seq_y: torch.Size([1, 2, 1]) tensor([[[4.],
         [nan]]])
window 2
seq_x: torch.Size([1, 2, 1]) tensor([[[3.],
         [4.]]])
seq_y: torch.Size([1, 2, 1]) tensor([[[4.],
         [nan]]])
window 3
seq_x: torch.Size([1, 2, 1]) tensor([[[5.],
         [nan]]])
seq_y: torch.Size([1, 2, 1]) tensor([[[nan],
         [nan]]])
window 4
seq_x: torch.Size([1, 2, 1]) tensor([[[5.],
         [nan]]])
seq_y: torch.Size([1, 2, 1]) tensor([[[nan],
         [nan]]])
window 5
seq_x: torch.Size([1, 2, 1]) tensor([[[6.],
         [7.]]])
seq_y: torch.Size([1, 2, 1]) tensor([[[7.],
         [8.]]])
window 6
seq_x: torch.Size([1, 2, 1]) tensor([[[7.],
         [8.]]])
seq_y: torch.Size([1, 2, 1]) tensor([[[8.],
         [9.]]])


In [9]:
from torch.utils.data import Dataset
from typing import Dict
import awkward as ak

class JaggedSlidingWindowDataset(Dataset):
    """
    Wraps JaggedAutoregressiveSequencer into a Dataset-like object you can feed to DataLoader.
    Expects data_dict keys: "features" (ak.Array) and optionally "time_features" (ak.Array).
    """

    def __init__(self, data_dict: Dict[str, ak.Array], seq_len, label_len, pred_len, stride=1, device=None):
        if "features" not in data_dict:
            raise ValueError("data_dict must contain 'features' key (awkward.Array)")
        features = data_dict["features"]
        time_features = data_dict.get("time_features", None)

        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.stride = stride

        self.seq = JaggedAutoregressiveSequencer(
            features=features,
            time_features=time_features,
            seq_len=seq_len,
            label_len=label_len,
            pred_len=pred_len,
            stride=stride,
            device=device,
        )

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        # idx might be a single int (DataLoader with batch_size=1 and no sampler) or a list of ints
        # We return the window corresponding to that index as a single-element batch for compatibility
        if isinstance(idx, int):
            seq_x, seq_y, seq_x_time, seq_y_time = self.seq.sequences_batch([idx])
            # squeeze batch dimension only if necessary? keep consistent: return tensors with batch dim 1
            return seq_x, seq_y, seq_x_time, seq_y_time
        elif isinstance(idx, list) or isinstance(idx, tuple):
            return self.seq.sequences_batch(list(idx))
        else:
            raise TypeError("idx must be int or list/tuple of ints")


In [22]:
# test_jagged.py
import awkward as ak
import torch

#features = ak.Array([[0, 1, 2], [3, 4], [5], [6, 7, 8, 9]])
# dummy time_features (same shape)
# timef = ak.Array([[0,0,0], [0,0], [0], [0,0,0,0]])
features = ak.Array([[[0,1,2], [3,4,5], [6,7,8]], [[9,10,11], [12,13,14]]])
timef = ak.Array([[0], [0]])


dataset = JaggedSlidingWindowDataset(
    data_dict={"features": features, "time_features": timef},
    seq_len=2,
    label_len=1,
    pred_len=1,
    stride=2,
    device=torch.device("cpu"),
)

print("Number of windows:", len(dataset))
# enumerate windows and their tensors
for i in range(len(dataset)):
    seq_x, seq_y, seq_x_time, seq_y_time = dataset[i]
    print("window", i)
    print("seq_x:",  seq_x)
    print("seq_y:",  seq_y)


Number of windows: 3
window 0
seq_x: tensor([[[0., 1., 2.],
         [3., 4., 5.]]])
seq_y: tensor([[[3., 4., 5.],
         [6., 7., 8.]]])


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

In [7]:
seq_x

tensor([[[7.],
         [8.]]])

In [37]:
import torch
from torch.utils.data import Dataset
import awkward as ak
import numpy as np


def pad_and_fill(
    array,
    target_len: int,
    num_features: int,
    do_pad: bool = True,
    pad_value: float | int = 0,
):
    """
    Pads an awkward array, correctly handling empty inputs to ensure consistent output shape.
    """
    if not do_pad:
        return array

    # If the input slice is empty, build a correctly shaped placeholder from scratch.
    if len(array) == 0:
        shape = (target_len, num_features)
        np_arr = np.full(shape, fill_value=pad_value, dtype=np.float32)
        return ak.from_numpy(np_arr)

    # Pad the non-empty array with None values
    padded = ak.pad_none(array, target_len, axis=0)
    # Use a template from the first element to create a correctly shaped padding record
    pad_record = ak.full_like(array[0], pad_value)
    return ak.fill_none(padded, pad_record)


class TimeSeriesSequencer:
    def __init__(self, features: ak.Array,
                 seq_len: int = 12, label_len: int = 6, pred_len: int = 1,
                 pad: bool = True, pad_value: float = 0.0):

        self.features = features
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.pad = pad
        self.pad_value = pad_value

        # Determine the number of features from the data
        self.num_features = ak.num(self.features[0, 0], axis=0) if len(features) > 0 else 0

        self._windows = self._build_windows()

    def _build_windows(self):
        """Creates a list of (sample_index, start_time_index) for each possible window."""
        windows = []
        for i, sample in enumerate(self.features):
            # The last valid starting point for seq_x to have at least one element.
            # We can create windows that need padding.
            max_start_t = len(sample) - 1
            for t in range(max_start_t + 1):
                 windows.append((i, t))
        return windows

    def __len__(self):
        return len(self._windows)

    def _ak_to_tensor(self, array, dtype=torch.float32):
        return torch.from_numpy(ak.to_numpy(array)).to(dtype)

    def _get_window_tensors_for(self, sample_idx: int, t: int):
        """Slices the data for a single window along the time dimension."""
        sample = self.features[sample_idx]

        # 1. Define window boundaries for seq_x and seq_y
        x_start = t
        x_end = t + self.seq_len
        y_start = t + self.seq_len - self.label_len
        y_end = y_start + self.label_len + self.pred_len

        # 2. Slice the awkward array. Awkward handles slicing past the end gracefully.
        seq_x_ak = sample[x_start:x_end]
        seq_y_ak = sample[y_start:y_end]

        # 3. Pad the slices to their required fixed lengths
        seq_x_padded = pad_and_fill(seq_x_ak, self.seq_len,
                                    num_features=self.num_features,
                                    pad_value=self.pad_value)
        seq_y_padded = pad_and_fill(seq_y_ak, self.label_len + self.pred_len,
                                    num_features=self.num_features,
                                    pad_value=self.pad_value)
        
        # 4. Convert to tensors
        seq_x = self._ak_to_tensor(seq_x_padded)
        seq_y = self._ak_to_tensor(seq_y_padded)

        return seq_x, seq_y

class SequenceDataset(Dataset):
    def __init__(self, sequencer: TimeSeriesSequencer):
        self.seq = sequencer

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        sample_idx, t = self.seq._windows[idx]
        return self.seq._get_window_tensors_for(sample_idx, t)



features = ak.Array([
    [[0, 1, 2], [3, 4, 5], [6, 7, 8]],
    [[9, 10, 11], [12, 13, 14]]
])

# Use parameters that match your desired slicing
# seq_len=2 takes 2 timesteps for x
# label_len=1 and pred_len=1 means y starts 1 step before the end of x and is 2 long
sequencer = TimeSeriesSequencer(
    features,
    seq_len=2,
    label_len=1,
    pred_len=1,
    pad=False,
    pad_value=-1
)

dataset = SequenceDataset(sequencer)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1)

# --- Print the first window (t=0) ---
seq_x, seq_y = dataset[0]
print("--- Window 0 (t=0) ---")
print("seq_x:\n", seq_x)
print("seq_y:\n", seq_y)


# --- Print a window that requires padding (t=2) ---
seq_x, seq_y = dataset[2]
print("\n--- Window 2 (t=2, requires padding) ---")
print("seq_x:\n", seq_x)
print("seq_y:\n", seq_y)

--- Window 0 (t=0) ---
seq_x:
 tensor([[0., 1., 2.],
        [3., 4., 5.]])
seq_y:
 tensor([[3., 4., 5.],
        [6., 7., 8.]])

--- Window 2 (t=2, requires padding) ---
seq_x:
 tensor([[6.0000e+00, 7.0000e+00, 8.0000e+00],
        [9.2234e+18, 9.2234e+18, 9.2234e+18]])
seq_y:
 tensor([[-1., -1., -1.],
        [-1., -1., -1.]])


In [ ]:
import awkward as ak
import numpy as np
import torch
from torch.utils.data import Dataset
from typing import Optional

# A simple logger/named transform input for the example
import logging
logger = logging.getLogger()
NamedTransformInput = dict
from lmetk.utils.awkward_utils import (
    ak_find_var_dims,
    ak_regularize_regular_axes,
    ragged_index_tuples,
)

from lmetk.transforms.base.multisource import find_singular_ragged_dim

TYPES_TO_ITERATE = (ak.Array, list)
def depth(arr):
    if (len(arr) == 0) or (not isinstance(arr[0], TYPES_TO_ITERATE)): 
        return 1
    else:
        return 1 + depth(arr[0])
    
class AdaptiveTimeSeriesSequencer:
    def __init__(self, features: ak.Array,
                 seq_len: int = 12, label_len: int = 6, pred_len: int = 1,
                 pad: bool = True, pad_value: float = 0.0):
        
        
        self.features = features
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.pad = pad # This flag now controls everything
        self.pad_value = pad_value

        self.ragged_dim = find_singular_ragged_dim({'features': self.features})
        if self.ragged_dim is None:
            raise ValueError("No ragged dimension found in the input arrays.")

        assert pad==False and depth(self.features)<3, 'In such a key there is nothing to pad'
        example_index = (0,) * (self.ragged_dim + 1)
        if self.pad:
            self.num_features = len(self.features[example_index])

        self._indices = self._build_indices()

    def _build_indices(self):
        """
        Creates a master list of index tuples.
        If padding is disabled, it only includes tuples for full, valid windows.
        """
        num_indexing_dims = self.ragged_dim + 1
        all_possible_indices = ragged_index_tuples(self.features, num_indexing_dims)

        if self.pad:
            # If padding is on, return an index for every possible start time
            return all_possible_indices
        else:
            # If padding is off, filter for complete windows only
            valid_indices = []
            for index_tuple in all_possible_indices:
                sample_idx = index_tuple[:self.ragged_dim]
                t = index_tuple[self.ragged_dim]
                
                # Get the length of the specific time series this index belongs to
                sample_len = len(self.features[sample_idx])
                
                # Calculate the required length for the full window (x and y)
                required_len = (t + self.seq_len - self.label_len) + (self.label_len + self.pred_len)

                # Only keep the index if the full window fits within the sequence
                if required_len <= sample_len:
                    valid_indices.append(index_tuple)
            return valid_indices

    def __len__(self):
        return len(self._indices)

    def _pad_and_fill(self, array, target_len):
        # This function is now only called when needed, but the logic remains robust
        if len(array) == 0:
            shape = (target_len, self.num_features)
            np_arr = np.full(shape, fill_value=self.pad_value, dtype=np.float32)
            return ak.from_numpy(np_arr)
        
        padded = ak.pad_none(array, target_len, axis=0)
        builder = ak.ArrayBuilder()
        pad_record = np.full(self.num_features, fill_value=self.pad_value, dtype=np.float32)
        for record in padded:
            builder.append(pad_record if record is None else record)
        return builder.snapshot()

    def _ak_to_tensor(self, array, dtype=torch.float32):
        return torch.from_numpy(ak.to_numpy(array)).to(dtype)

    def get_window_for_index(self, index_tuple: tuple):
        sample_idx = index_tuple[:self.ragged_dim]
        t = index_tuple[self.ragged_dim]
        sample = self.features[sample_idx]

        x_start, x_end = t, t + self.seq_len
        y_start = t + self.seq_len - self.label_len
        y_end = y_start + self.label_len + self.pred_len
        
        seq_x_ak = sample[x_start:x_end]
        seq_y_ak = sample[y_start:y_end]
        
        # When pad=False, the arrays passed here will already be the correct size,
        # so the padding function will effectively just convert them.
        seq_x_padded = self._pad_and_fill(seq_x_ak, self.seq_len)
        seq_y_padded = self._pad_and_fill(seq_y_ak, self.label_len + self.pred_len)
        
        return self._ak_to_tensor(seq_x_padded), self._ak_to_tensor(seq_y_padded)

# The Dataset wrapper remains the same
class SequenceDataset(Dataset):
    def __init__(self, sequencer: AdaptiveTimeSeriesSequencer):
        self.seq = sequencer
    def __len__(self):
        return len(self.seq)
    def __getitem__(self, idx):
        index_tuple = self.seq._indices[idx]
        return self.seq.get_window_for_index(index_tuple)

In [ ]:
features_complex = ak_regularize_regular_axes(ak.Array([
    [ # Sample 0
        [0, 1], [2, 3]         # Sub-sample (0,0)
    ],
    [ # Sample 1
        [10, 11]                # Sub-sample (1,0)
    ],
    [
        [4, 5], [6, 7], [8, 9]
    ]
]))
print('PADDING is set to FALSE')
# The sequencer automatically detects ragged_dim = 2
sequencer_complex = AdaptiveTimeSeriesSequencer(
    features = features_complex,
    seq_len=2, label_len=1, pred_len=1,  pad=False, pad_value=-1
)

dataset_complex = SequenceDataset(sequencer_complex)
# The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
for i in range(dataset_complex.__len__()):
    seq_x, seq_y = dataset_complex[i]

    print("\n## Complex Data (N * var * C)")
    print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
    print("seq_x:\n", seq_x)
    print("seq_y:\n", seq_y)


print('PADDING is set to TRUE')

# The sequencer automatically detects ragged_dim = 2
sequencer_complex = AdaptiveTimeSeriesSequencer(
    features = features_complex,
    seq_len=2, label_len=1, pred_len=1,  pad=True, pad_value=-1
)

dataset_complex = SequenceDataset(sequencer_complex)
# The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
for i in range(dataset_complex.__len__()):
    seq_x, seq_y = dataset_complex[i]

    print("\n## Complex Data (N * var * C)")
    print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
    print("seq_x:\n", seq_x)
    print("seq_y:\n", seq_y)




PADDING is set to FALSE

## Complex Data (N * M * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[4., 5.],
        [6., 7.]])
seq_y:
 tensor([[6., 7.],
        [8., 9.]])
PADDING is set to TRUE


In [33]:
features_complex = ak_regularize_regular_axes(ak.Array([
    [ # Sample 0
        [0, 1], [2, 3]         # Sub-sample (0,0)
    ],
    [ # Sample 1
        [10, 11]                # Sub-sample (1,0)
    ],
    [
        [4, 5], [6, 7], [8, 9]
    ]
]))
print('PADDING is set to FALSE')
# The sequencer automatically detects ragged_dim = 2
sequencer_complex = AdaptiveTimeSeriesSequencer(
    features = features_complex,
    seq_len=2, label_len=1, pred_len=1,  pad=True, pad_value=-1
)

dataset_complex = SequenceDataset(sequencer_complex)
# The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
for i in range(dataset_complex.__len__()):
    seq_x, seq_y = dataset_complex[i]

    print("\n## Complex Data (N * var * C)")
    print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
    print("seq_x:\n", seq_x)
    print("seq_y:\n", seq_y)


print('PADDING is set to TRUE')

PADDING is set to FALSE

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[0., 1.],
        [2., 3.]])
seq_y:
 tensor([[ 2.,  3.],
        [-1., -1.]])

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[ 2.,  3.],
        [-1., -1.]])
seq_y:
 tensor([[-1., -1.],
        [-1., -1.]])

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[10., 11.],
        [-1., -1.]])
seq_y:
 tensor([[-1., -1.],
        [-1., -1.]])

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[4., 5.],
        [6., 7.]])
seq_y:
 tensor([[6., 7.],
        [8., 9.]])

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[6., 7.],
        [8., 9.]])
seq_y:
 tensor([[ 8.,  9.],
        [-1., -1.]])

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[ 8.,  9.],
        [-1., -1.]])
seq_y:
 tensor([[-1., -1.],
        [-1., -1.]])
PADDING is set to TRUE


In [46]:
features_complex = ak.Array([[0, 1, 3, 4], [5,6, 7], [8, 9,10,11,12,13,14]])
print('PADDING is set to FALSE')
# The sequencer automatically detects ragged_dim = 2
sequencer_complex = AdaptiveTimeSeriesSequencer(
    {'features': features_complex},
    seq_len=2, label_len=1, pred_len=1,  pad=True, pad_value=-1
)

dataset_complex = SequenceDataset(sequencer_complex)
# The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
for i in range(dataset_complex.__len__()):
    seq_x, seq_y = dataset_complex[i]

    print("\n## Complex Data (N * var * C)")
    print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
    print("seq_x:\n", seq_x)
    print("seq_y:\n", seq_y)


print('PADDING is set to TRUE')

PADDING is set to FALSE

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 features_x
seq_y:
 features_y

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 features_x
seq_y:
 features_y


TypeError: Conversion of irreducible unions to backend arrays is not supported.

In [36]:
features_complex

<Array [[[0, 1, 3, 4]], ..., [[8, 9, ..., 14]]] type='3 * var * var * int64'>

features_complex

<Array [[[0, 1], [2, 3]], ..., [[4, 5], ..., [...]]] type='3 * var * 2 * int64'>

In [ ]:
import awkward as ak
import numpy as np
import torch
from torch.utils.data import Dataset
from typing import Optional, Tuple

# Assuming these functions are imported from your local library
# For this example, I've included the necessary helpers.
NamedTransformInput = dict

class AdaptiveTimeSeriesSequencer:
    """
    Creates time-series windows from a single ragged Awkward Array.
    """
    def __init__(self, features: ak.Array,
                 seq_len: int = 12, label_len: int = 6, pred_len: int = 1,
                 pad: bool = True, pad_value: float = 0.0):
        
        self.features = features
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.pad = pad
        self.pad_value = pad_value

        self.ragged_dim = find_singular_ragged_dim({'features': self.features})
        if self.ragged_dim is None:
            raise ValueError("No ragged dimension found in the input array.")

        example_index = (0,) * (self.ragged_dim + 1)
        example_element = self.features[example_index]
        if isinstance(example_element, (int, float, np.number)):
            self.num_features = 1
        else:
            self.num_features = len(example_element)

        self._indices = self._build_indices()

    def _build_indices(self) -> list[tuple]:
        """Builds a master list of indices, filtering for complete windows if padding is off."""
        num_indexing_dims = self.ragged_dim + 1
        all_possible_indices = ragged_index_tuples(self.features, num_indexing_dims)
        if self.pad:
            return all_possible_indices
        
        valid_indices = []
        for index_tuple in all_possible_indices:
            sample_idx = index_tuple[:self.ragged_dim]
            t = index_tuple[self.ragged_dim]
            sample_len = len(self.features[sample_idx])
            required_len = (t + self.seq_len - self.label_len) + (self.label_len + self.pred_len)
            if required_len <= sample_len:
                valid_indices.append(index_tuple)
        return valid_indices

    def __len__(self) -> int:
        return len(self._indices)

    def _pad_and_fill(self, array: ak.Array, target_len: int) -> ak.Array:
        """Pads a sliced array to the target length."""
        if len(array) == 0:
            shape = (target_len, self.num_features)
            return ak.from_numpy(np.full(shape, self.pad_value, dtype=np.float32))
        
        padded = ak.pad_none(array, target_len, axis=0)
        builder = ak.ArrayBuilder()
        pad_record = np.full(self.num_features, self.pad_value, dtype=np.float32)
        for record in padded:
            # Here, we must ensure scalars are appended in a way that matches the pad_record structure
            if record is None:
                builder.append(pad_record)
            elif self.num_features == 1 and isinstance(record, (int, float, np.number)):
                builder.append(np.array([record], dtype=np.float32)) # Wrap scalar
            else:
                builder.append(record)
        return builder.snapshot()

    def _ak_to_tensor(self, array: ak.Array) -> torch.Tensor:
        """Converts a finalized Awkward array to a PyTorch tensor."""
        np_arr = ak.to_numpy(array)
        if self.num_features == 1 and np_arr.ndim == 1:
            np_arr = np_arr.reshape(-1, 1)
        return torch.from_numpy(np_arr).to(torch.float32)

    def get_window_for_index(self, index_tuple: tuple) -> Tuple[torch.Tensor, torch.Tensor]:
        """Extracts, pads (if enabled), and converts a single window."""
        sample_idx = index_tuple[:self.ragged_dim]
        t = index_tuple[self.ragged_dim]
        sample = self.features[sample_idx]

        x_start, x_end = t, t + self.seq_len
        y_start = t + self.seq_len - self.label_len
        y_end = y_start + self.label_len + self.pred_len
        
        seq_x_ak = sample[x_start:x_end]
        seq_y_ak = sample[y_start:y_end]
        
        # Only call the padding function when self.pad is True.
        if self.pad:
            seq_x_ak = self._pad_and_fill(seq_x_ak, self.seq_len)
            seq_y_ak = self._pad_and_fill(seq_y_ak, self.label_len + self.pred_len)
        
        return self._ak_to_tensor(seq_x_ak), self._ak_to_tensor(seq_y_ak)

class SequenceDataset(Dataset):
    """Standard PyTorch Dataset wrapper."""
    def __init__(self, sequencer: AdaptiveTimeSeriesSequencer):
        self.seq = sequencer

    def __len__(self) -> int:
        return len(self.seq)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        index_tuple = self.seq._indices[idx]
        return self.seq.get_window_for_index(index_tuple)

1
torch.Size([3, 1]) torch.Size([2, 1])


In [ ]:
import awkward as ak
import numpy as np
import torch
from torch.utils.data import Dataset
from typing import Optional, Tuple
import numpy as np
import torch
from torch.utils.data import Dataset
from typing import Optional

# A simple logger/named transform input for the example
import logging

logger = logging.getLogger()

from lmetk.utils.awkward_utils import (
    ak_find_var_dims,
    ak_regularize_regular_axes,
    ragged_index_tuples,
)

from lmetk.transforms.base.multisource import find_singular_ragged_dim


class AdaptiveTimeSeriesSequencer:
    """
    Creates time-series windows from a single ragged Awkward Array.
    """

    def __init__(
        self,
        features: ak.Array,
        seq_len: int,
        label_len: int,
        pred_len: int,
        pad: bool = False,
        pad_value: float = 0.0,
    ):

        self.features = features
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.pad = pad
        self.pad_value = pad_value

        self.ragged_dim = find_singular_ragged_dim({"features": self.features})
        if self.ragged_dim is None:
            raise ValueError("No ragged dimension found in the input array.")

        if pad:
            raise NotImplementedError(
                "Padding functionality is not checked yet."
                "Please verify correctness before using in production."
                "The masking functionality is not implemented yet."
            )

        example_index = (0,) * (self.ragged_dim + 1)
        example_element = self.features[example_index]
        if isinstance(example_element, (int, float, np.number)):
            self.num_features = 1
        else:
            self.num_features = len(example_element)

        self._indices = self._build_indices()

    def _build_indices(self) -> list[tuple]:
        """
        Build a master list of indices.

        Logic:
        1) Drop samples that are too short to contain a single window.
        2) For each remaining sample, keep only those *starting* timesteps
            that can generate a complete (seq_x, seq_y) window.
        """
        num_indexing_dims = self.ragged_dim + 1

        # 1) Drop sequences that are too short
        lengths = ak.num(self.features, axis=self.ragged_dim)
        min_required = self.seq_len + self.pred_len
        keep_mask = lengths >= min_required
        features_kept = self.features[keep_mask]
        lengths_kept = lengths[keep_mask]

        # 2) For each sample, retain only valid *starting* indices
        # Valid starts: t ∈ [0, t_max] where t_max = L - (seq_len + pred_len)
        trimmed_samples = []
        for sample, L in zip(features_kept, lengths_kept):
            t_max = L - (self.seq_len + self.pred_len)
            # Keep just the range of starting indices 0..t_max inclusive
            # (ragged_index_tuples enumerates positions along this axis)
            cut_len = max(0, t_max + 1)
            trimmed_samples.append(sample[:cut_len])

        trimmed_array = ak.Array(trimmed_samples)

        # 3) Enumerate all valid (sample_idx, t, ...) tuples
        all_possible_indices = ragged_index_tuples(trimmed_array, num_indexing_dims)
        return all_possible_indices

    def __len__(self) -> int:
        return len(self._indices)

    def _pad_and_fill(self, array: ak.Array, target_len: int) -> ak.Array:
        """Pads a sliced array to the target length."""
        if len(array) == 0:
            shape = (target_len, self.num_features)
            return ak.from_numpy(np.full(shape, self.pad_value, dtype=np.float32))

        padded = ak.pad_none(array, target_len, axis=0)
        builder = ak.ArrayBuilder()
        pad_record = np.full(self.num_features, self.pad_value, dtype=np.float32)
        for record in padded:
            # Here, we must ensure scalars are appended in a way that matches the pad_record structure
            if record is None:
                builder.append(pad_record)
            elif self.num_features == 1 and isinstance(record, (int, float, np.number)):
                builder.append(np.array([record], dtype=np.float32))  # Wrap scalar
            else:
                builder.append(record)
        return builder.snapshot()

    def _ak_to_tensor(self, array: ak.Array) -> torch.Tensor:
        """Converts a finalized Awkward array to a PyTorch tensor."""
        np_arr = ak.to_numpy(array)
        if self.num_features == 1 and np_arr.ndim == 1:
            np_arr = np_arr.reshape(-1, 1)
        return torch.from_numpy(np_arr).to(torch.float32)

    def get_window_for_index(
        self, index_tuple: tuple
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Extracts, pads (if enabled), and converts a single window."""
        sample_idx = index_tuple[: self.ragged_dim]
        t = index_tuple[self.ragged_dim]
        sample = self.features[sample_idx]

        x_start, x_end = t, t + self.seq_len
        y_start = t + self.seq_len - self.label_len
        y_end = y_start + self.label_len + self.pred_len

        seq_x_ak = sample[x_start:x_end]
        seq_y_ak = sample[y_start:y_end]

        # --- KEY FIX IS HERE ---
        # Only call the padding function when self.pad is True.
        if self.pad:
            seq_x_ak = self._pad_and_fill(seq_x_ak, self.seq_len)
            seq_y_ak = self._pad_and_fill(seq_y_ak, self.label_len + self.pred_len)

        return self._ak_to_tensor(seq_x_ak), self._ak_to_tensor(seq_y_ak)


class SequenceDataset(Dataset):
    """Standard PyTorch Dataset wrapper."""

    def __init__(self, sequencer: AdaptiveTimeSeriesSequencer):
        self.seq = sequencer

    def __len__(self) -> int:
        return len(self.seq)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        index_tuple = self.seq._indices[idx]
        return self.seq.get_window_for_index(index_tuple)


if __name__ == "__main__":

    features_complex = ak.Array([[0, 1, 3, 4], [5, 6, 7], [8, 9, 10, 11, 12, 13, 14]])
    print("PADDING is set to FALSE")
    # The sequencer automatically detects ragged_dim = 2
    sequencer_complex = AdaptiveTimeSeriesSequencer(
        features=features_complex,
        seq_len=3,
        label_len=1,
        pred_len=1,
        pad=False,
        pad_value=-1,
    )

    dataset_complex = SequenceDataset(sequencer_complex)
    # The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
    for i in range(dataset_complex.__len__()):
        seq_x, seq_y = dataset_complex[i]

        print("\n## Complex Data (N * var )")
        print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
        print("seq_x:\n", seq_x)
        print("seq_y:\n", seq_y)

    
    print("PADDING is set to FALSE")
    features_complex = ak_regularize_regular_axes(
        ak.Array(
            [
                [[0, 1], [2, 3]],  # Sample 0  # Sub-sample (0,0)
                [[10, 11]],  # Sample 1  # Sub-sample (1,0)
                [[4, 5], [6, 7], [8, 9]],
            ]
        )
    )
    # The sequencer automatically detects ragged_dim = 2
    sequencer_complex = AdaptiveTimeSeriesSequencer(
        features=features_complex,
        seq_len=2,
        label_len=1,
        pred_len=1,
        pad=False,
        pad_value=-1,
    )

    dataset_complex = SequenceDataset(sequencer_complex)
    # The 3rd overall timestep in the data corresponds to index tuple (0, 1, 1)
    for i in range(dataset_complex.__len__()):
        seq_x, seq_y = dataset_complex[i]

        print("\n## Complex Data (N * var * C)")
        print(f"Detected ragged dimension: {sequencer_complex.ragged_dim}")
        print("seq_x:\n", seq_x)
        print("seq_y:\n", seq_y)

    

PADDING is set to FALSE

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([[0.],
        [1.],
        [3.]])
seq_y:
 tensor([[3.],
        [4.]])

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([[5.],
        [6.],
        [7.]])
seq_y:
 tensor([[7.]])

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([[6.],
        [7.]])
seq_y:
 tensor([], size=(0, 1))

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([[7.]])
seq_y:
 tensor([], size=(0, 1))

## Complex Data (N * var )
Detected ragged dimension: 1
seq_x:
 tensor([], size=(0, 1))
seq_y:
 tensor([], size=(0, 1))
PADDING is set to FALSE

## Complex Data (N * var * C)
Detected ragged dimension: 1
seq_x:
 tensor([[0., 1.],
        [2., 3.]])
seq_y:
 tensor([[2., 3.]])
